# Data load & Sanity Check

TODO: Move some of the training stuff to src/donderformer/training if applicable

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from donderformer.data.features import FeatureExtractor
from donderformer.paths import DataPaths, outputs_dir

# Data root: DONDER_DATA env var, else <repo>/data. Use DataPaths.toy() for a quick local run.
paths = DataPaths()
SPLIT = "oni_v1"                              # folder under splits/, made by 01_make_split
extractor = FeatureExtractor(frame_ms=23)     # cache made by 02_extract_features
WINDOW = dict(audio_frames=16, history_frames=15, target_frames=4)   # TaikoNation defaults

RUN_NAME = f"taikonation_{SPLIT}_{extractor.id}"
CHECKPOINT = outputs_dir() / "checkpoints" / f"{RUN_NAME}.pt"
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
DATA_INFO = {"split": SPLIT, "features": extractor.id, **WINDOW}

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# local setup
# device = torch.device(
#     "mps" if torch.backends.mps.is_available()
#     else "cuda" if torch.cuda.is_available()
#     else "cpu"
# )

# print("Device:", device)

# PACE - cuda expected
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

Device: mps


### Model Pipeline

**One input window**  
↓  
**16 audio frames (16 × 80) + 15 previous note frames (15 × 7)**  
↓  
**CNN extracts features from the audio**  
↓  
**Audio features are combined with previous note information**  
↓  
**LSTM learns temporal/rhythmic patterns**  
↓  
**Predict 4 timesteps (4 × 7 classes)**

In [ ]:
class TaikoNationBaseline(nn.Module):
    def __init__(self):
        super().__init__()

        # Split the 1385 inputs; split them back into audio and notes
        self.conv1 = nn.Conv1d(80, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(2)

        self.audio_fc = nn.Linear(32 * 4, 128)

        self.lstm1 = nn.LSTM(
            input_size=8,
            hidden_size=64,
            batch_first=True
        )

        self.lstm2 = nn.LSTM(
            input_size=64,
            hidden_size=64,
            batch_first=True
        )

        self.output = nn.Linear(64, 28)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.8)

    # send the audio through the CNN to find useful patters in the audio features
    def forward(self, x):
        audio = x[:, :1280].reshape(-1, 16, 80)
        notes = x[:, 1280:].reshape(-1, 15, 7)

        audio = audio.transpose(1, 2)

        audio = self.relu(self.conv1(audio))
        audio = self.dropout(audio)
        audio = self.pool1(audio)

        audio = self.relu(self.conv2(audio))
        audio = self.pool2(audio)

        audio = audio.flatten(1)
        audio = self.relu(self.audio_fc(audio))
        audio = audio.reshape(-1, 16, 8)

        note_bias = torch.ones(
            notes.shape[0], 15, 1,
            device=notes.device,
            dtype=notes.dtype
        )

        note_history = torch.cat([notes, note_bias], dim=2)

        current_placeholder = torch.ones(
            notes.shape[0], 1, 8,
            device=notes.device,
            dtype=notes.dtype
        )

        note_context = torch.cat(
            [note_history, current_placeholder],
            dim=1
        )

        # Combine audio and previous-note information
        combined = audio * note_context

        combined, _ = self.lstm1(combined)
        combined = self.relu(combined)
        combined = self.dropout(combined)

        # LSTM learns the temporal pattern
        combined, _ = self.lstm2(combined)
        combined = self.relu(combined)
        combined = self.dropout(combined)

        final = combined[:, -1, :]
        logits = self.output(final)

        return logits.reshape(-1, 4, 7)

# sanity check

In [ ]:
model = TaikoNationBaseline().to(device)
from donderformer.data.dataset import TaikoDataset
sample_ds = TaikoDataset(paths, SPLIT, extractor, "train", **WINDOW)
X, _ = sample_ds.chart_windows(sample_ds.chart_ids[0])
sample = torch.from_numpy(X[:8]).to(device)
with torch.no_grad():
    output = model(sample)
print("Input:", sample.shape, "Output:", output.shape)
assert output.shape == (sample.shape[0], 4, 7)

# data loader

Songs are loaded one at a time (see `donderformer.data.dataset`) because the full set is ~2k charts.

In [ ]:
from torch.utils.data import DataLoader
from donderformer.data.dataset import SongBatchSampler, TaikoDataset

train_dataset = TaikoDataset(paths, SPLIT, extractor, "train", **WINDOW)
val_dataset = TaikoDataset(paths, SPLIT, extractor, "validation", **WINDOW)
test_dataset = TaikoDataset(paths, SPLIT, extractor, "test", **WINDOW)
train_loader = DataLoader(train_dataset,
    batch_sampler=SongBatchSampler(train_dataset, 16, shuffle=True), num_workers=0)
val_loader = DataLoader(val_dataset,
    batch_sampler=SongBatchSampler(val_dataset, 16), num_workers=0)
for name, ds in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    print(f"{name}: {len(ds.chart_ids)} charts, {len(ds)} windows")
print("Train batches:", len(train_loader), "Val batches:", len(val_loader))

# loss + optimizer

In [ ]:
# measures how wrong the model's predictions are compared withthe correct note classes
criterion = nn.CrossEntropyLoss()

# changes the model's weights to reduce the loss; Adam is a popular optimizer that works well in many situations
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-5
)

# training & validation functions

In [ ]:
# predict -> calculate loss -> backpropagate -> update weights
def train_epoch(model, loader):
    model.train()

    total_loss = 0.0

    for X, Y in loader:
        X = X.to(device)
        Y = Y.to(device)

        # before processing the new batch, we clear the gradients from the previous batch.
        optimizer.zero_grad()

        logits = model(X)
        targets = Y.argmax(dim=2)

        loss = criterion(
            logits.reshape(-1, 7),
            targets.reshape(-1)
        )

        # learn from loss; how did each model weight contribute to the loss? how should we change the weights to reduce the loss?
        loss.backward()
        # use gradients to update the model's weights  
        optimizer.step()

        total_loss += loss.item()

    # returns the average loss across all batches in the epoch
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()

    total_loss = 0.0

    with torch.no_grad():
        for X, Y in loader:
            X = X.to(device)
            Y = Y.to(device)

            logits = model(X)
            targets = Y.argmax(dim=2)

            loss = criterion(
                logits.reshape(-1, 7),
                targets.reshape(-1)
            )

            total_loss += loss.item()

    return total_loss / len(loader)

# Train 10 epochs

In [10]:
# first 10 epochs
train_losses = []
val_losses = []

for epoch in range(10):
    train_loss = train_epoch(model, train_loader)
    val_loss = evaluate(model, val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

Epoch 01 | Train Loss: 1.5563 | Val Loss: 0.8068
Epoch 02 | Train Loss: 0.9686 | Val Loss: 0.5228
Epoch 03 | Train Loss: 0.8012 | Val Loss: 0.4207
Epoch 04 | Train Loss: 0.7301 | Val Loss: 0.3776
Epoch 05 | Train Loss: 0.6933 | Val Loss: 0.3561
Epoch 06 | Train Loss: 0.6705 | Val Loss: 0.3429
Epoch 07 | Train Loss: 0.6515 | Val Loss: 0.3335
Epoch 08 | Train Loss: 0.6370 | Val Loss: 0.3257
Epoch 09 | Train Loss: 0.6239 | Val Loss: 0.3186
Epoch 10 | Train Loss: 0.6083 | Val Loss: 0.3117


# Second stage - fine tuning

Replicate paper's training procedure with batch size 1 and learning rate 5e-6

In [ ]:
train_loader_finetune = DataLoader(
    train_dataset,
    batch_sampler=SongBatchSampler(train_dataset, 1, shuffle=True),
    num_workers=0
)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-6)
for epoch in range(4):
    train_loss = train_epoch(model, train_loader_finetune)
    val_loss = evaluate(model, val_loader)
    print(f"Fine-tune Epoch {epoch + 1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

# save model

In [ ]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "train_losses": train_losses,
        "val_losses": val_losses,
        "data": DATA_INFO
    },
    CHECKPOINT
)

print("Saved.")

# Make predictions on the test song

### Test Prediction Pipeline

**Trained Model + Test Audio/Note Windows**  
↓  
**Model Prediction**  
↓  
**Raw Scores (Logits)**  
↓  
**Softmax**  
↓  
**Probabilities for Each of the 7 Note Classes**  
↓  
**Used for Chart Generation**

In [ ]:
# Inference is now performed song-by-song in Cell 31, in small GPU batches.
model.eval()
INFERENCE_BATCH_SIZE = 64
print("Ready to evaluate", len(test_dataset.chart_ids), "test charts")

# Combining the overlapping predictions from different windows to produce one final 7-class probability distribution for each 23 ms timestep.

In [ ]:
def aggregate_overlapping_predictions(probs):
    # probs: (number_of_windows, 4, 7), from ONE song only
    n = len(probs)
    if n == 0:
        return np.empty((0, 7), dtype=np.float32)
    aggregated = np.zeros((n + 3, 7), dtype=np.float32)
    counts = np.zeros(n + 3, dtype=np.float32)
    for j in range(4):
        aggregated[j:j+n] += probs[:, j, :]
        counts[j:j+n] += 1
    return aggregated / counts[:, None]

# Generating chart

Takes the final probability distribution for every 23 ms timestep and turns it into an actual generated chart.

In [ ]:
CLASS_NAMES = ["none", "don", "ka", "DON", "KA", "drumroll", "denden"]
rng = np.random.default_rng(42)

def sample_chart(aggregated):
    return np.array([rng.choice(7, p=p / p.sum()) for p in aggregated],
                    dtype=np.int64)

# Target vs. Model generated

In [ ]:
def reconstruct_true_sequence(Y):
    # Y: (number_of_windows, 4, 7), for ONE song only
    targets = Y.argmax(axis=2)
    n = len(targets)
    if n == 0:
        return np.empty(0, dtype=np.int64)
    sequence = np.empty(n + 3, dtype=np.int64)
    sequence[:n] = targets[:, 0]
    sequence[n:] = targets[-1, 1:4]
    return sequence

# Onset evaluation

Regardless of note type like Don and Ka, this measures generated notes occur near the ground-truth note timings (within +- 23ms).

In [ ]:
def onset_metrics(true_seq, pred_seq, tolerance=1):
    true_onsets = np.where(true_seq != 0)[0]
    pred_onsets = np.where(pred_seq != 0)[0]

    matched_true = set()
    matches = 0

    for pred in pred_onsets:
        candidates = [
            t for t in true_onsets
            if abs(t - pred) <= tolerance and t not in matched_true
        ]

        if candidates:
            best = min(candidates, key=lambda t: abs(t - pred))
            matched_true.add(best)
            matches += 1

    precision = matches / len(pred_onsets) if len(pred_onsets) else 0
    recall = matches / len(true_onsets) if len(true_onsets) else 0

    if precision + recall:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0

    return precision, recall, f1, matches

# Onset + type eval

In [ ]:
def joint_onset_type_metrics(true_seq, pred_seq, tolerance=1):
    true_onsets = np.where(true_seq != 0)[0]
    pred_onsets = np.where(pred_seq != 0)[0]

    matched_true = set()
    matches = 0

    for pred in pred_onsets:
        candidates = [
            t for t in true_onsets
            if abs(t - pred) <= tolerance
            and true_seq[t] == pred_seq[pred]
            and t not in matched_true
        ]

        if candidates:
            best = min(candidates, key=lambda t: abs(t - pred))
            matched_true.add(best)
            matches += 1

    precision = matches / len(pred_onsets) if len(pred_onsets) else 0
    recall = matches / len(true_onsets) if len(true_onsets) else 0

    if precision + recall:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0

    return precision, recall, f1, matches

def score_from_counts(tp, predicted, actual):
    precision = tp / predicted if predicted else 0.0
    recall = tp / actual if actual else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

# Micro-averaged evaluation: accumulate counts across songs, never across boundaries.
model.eval()
total_true = total_pred = total_onset_matches = total_joint_matches = 0
true_class_counts = np.zeros(7, dtype=np.int64)
pred_class_counts = np.zeros(7, dtype=np.int64)

for song_index, song_id in enumerate(test_dataset.chart_ids, start=1):
    X, Y = test_dataset.chart_windows(song_id)
    if len(X) == 0:
        continue
    song_probs = []
    with torch.no_grad():
        for start in range(0, len(X), INFERENCE_BATCH_SIZE):
            batch = torch.from_numpy(
                X[start:start + INFERENCE_BATCH_SIZE].astype(np.float32)
            ).to(device)
            logits = model(batch)
            song_probs.append(torch.softmax(logits, dim=2).cpu().numpy())
    probs = np.concatenate(song_probs, axis=0)
    generated = sample_chart(aggregate_overlapping_predictions(probs))
    true_sequence = reconstruct_true_sequence(Y)
    assert len(generated) == len(true_sequence)

    _, _, _, onset_matches = onset_metrics(true_sequence, generated, tolerance=1)
    _, _, _, joint_matches = joint_onset_type_metrics(true_sequence, generated, tolerance=1)
    total_onset_matches += onset_matches
    total_joint_matches += joint_matches
    total_true += np.count_nonzero(true_sequence)
    total_pred += np.count_nonzero(generated)
    true_class_counts += np.bincount(true_sequence, minlength=7)
    pred_class_counts += np.bincount(generated, minlength=7)
    if song_index % 20 == 0:
        print(f"Evaluated {song_index}/{len(test_dataset.chart_ids)} charts")

precision, recall, f1 = score_from_counts(total_onset_matches, total_pred, total_true)
precision_type, recall_type, f1_type = score_from_counts(total_joint_matches, total_pred, total_true)
print(f"Onset: precision={precision:.4f}, recall={recall:.4f}, F1={f1:.4f}")
print(f"Onset + type: precision={precision_type:.4f}, recall={recall_type:.4f}, F1={f1_type:.4f}")
for name, actual, predicted in zip(CLASS_NAMES, true_class_counts, pred_class_counts):
    print(f"{name:10s} true={actual:8d} generated={predicted:8d}")

In [ ]:
results = {
    "onset_precision": float(precision),
    "onset_recall": float(recall),
    "onset_f1": float(f1),
    "joint_precision": float(precision_type),
    "joint_recall": float(recall_type),
    "joint_f1": float(f1_type)
}

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "results": results,
        "data": DATA_INFO
    },
    CHECKPOINT
)

print("Model and evaluation results saved.")
print(results)